<a href="https://colab.research.google.com/github/ThLanBot42/DHU_AI_THL_CV_COMP_COMPETITION/blob/main/Firegazing_V4_0_helper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 【新 Notebook - YOLO11x 异构训练】修复版
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics --upgrade  # 确保最新版支持 YOLO11

from ultralytics import YOLO
import os

ROOT = '/content/drive/MyDrive/Wheat'

# ========== 正确名称：yolo11x.pt（没有 v）==========
model = YOLO('yolo11x.pt')

model.train(
    data=f'{ROOT}/data_fixed.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    lr0=0.005,
    lrf=0.01,
    patience=50,

    # 更强的麦穗增强
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.4,
    degrees=25.0,
    translate=0.3,
    scale=0.7,
    shear=3.0,
    perspective=0.001,
    flipud=0.5,
    fliplr=0.5,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,

    box=7.5, cls=0.5, dfl=1.5,
    project=f'{ROOT}/4_output',
    name='A_yolo11x_single',
    exist_ok=True
)

print("✅ YOLO11x 训练完成")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.4, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Wheat/data_fixed.yaml, degrees=25.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, ex

In [8]:
!pip install -q ultralytics torch torchvision --upgrade

from google.colab import drive
drive.mount('/content/drive')

import os, csv
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import torch
from torchvision.ops import nms

ROOT = '/content/drive/MyDrive/Wheat'
A_TEST = Path(ROOT) / '0_raw/images/A_test'
MODEL = Path(ROOT) / '4_output/A_yolo11x_single/weights/best.pt'
SAVE_CSV = Path(ROOT) / '4_output/submission_A_yolo11x_final.csv'

model = YOLO(str(MODEL))
print(f"模型: {MODEL}")

a_test = sorted(A_TEST.glob('*.png'))
print(f"测试图: {len(a_test)}张")

SCALES = [640, 768, 896]
CONF = 0.001
IOU_NMS = 0.5

all_results = []

for idx, img_path in enumerate(a_test):
    if idx % 100 == 0:
        print(f"{idx}/{len(a_test)}")

    image_id = img_path.stem
    with Image.open(img_path) as img:
        img_w, img_h = img.size

    boxes_list, scores_list = [], []

    for scale in SCALES:
        r = model(str(img_path), imgsz=scale, conf=CONF, iou=IOU_NMS, augment=True, verbose=False)[0]
        if len(r.boxes) > 0:
            boxes = r.boxes.xyxy.cpu()
            scores = r.boxes.conf.cpu()
            if len(boxes) > 300:
                topk = torch.topk(scores, 300)
                boxes = boxes[topk.indices]
                scores = scores[topk.indices]
            boxes_list.append(boxes)
            scores_list.append(scores)

    if len(boxes_list) > 0:
        all_boxes = torch.cat(boxes_list, dim=0)
        all_scores = torch.cat(scores_list, dim=0)
        keep = nms(all_boxes, all_scores, IOU_NMS)
        final_boxes = all_boxes[keep]
        final_scores = all_scores[keep]
    else:
        final_boxes = torch.empty((0, 4))
        final_scores = torch.empty(0)

    for i in range(len(final_boxes)):
        x1, y1, x2, y2 = final_boxes[i].tolist()
        x1 = max(0.0, min(x1, img_w))
        y1 = max(0.0, min(y1, img_h))
        x2 = max(0.0, min(x2, img_w))
        y2 = max(0.0, min(y2, img_h))
        w = x2 - x1
        h = y2 - y1

        # ═══ 核心修复：官网要求宽高必须严格大于0 ═══
        if w > 0 and h > 0:
            xc = (x1 + w / 2.0) / img_w
            yc = (y1 + h / 2.0) / img_h
            nw = w / img_w
            nh = h / img_h
            conf = float(final_scores[i])
            xc = max(0.0, min(1.0, xc))
            yc = max(0.0, min(1.0, yc))
            nw = max(0.0, min(1.0, nw))
            nh = max(0.0, min(1.0, nh))
            all_results.append({"image_id": image_id, "class_id": 0, "x_center": f"{xc:.6f}", "y_center": f"{yc:.6f}", "width": f"{nw:.6f}", "height": f"{nh:.6f}", "confidence": f"{conf:.6f}"})

SAVE_CSV.parent.mkdir(parents=True, exist_ok=True)

fieldnames = ['image_id', 'class_id', 'x_center', 'y_center', 'width', 'height', 'confidence']
with open(SAVE_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, lineterminator='\r\n')
    writer.writeheader()
    writer.writerows(all_results)

print(f"\nCSV: {SAVE_CSV}")
print(f"总行数: {len(all_results)}")

# 自检：确保没有 width<=0 或 height<=0
import pandas as pd
df = pd.read_csv(SAVE_CSV)
bad = df[(df['width'] <= 0) | (df['height'] <= 0)]
print(f"宽高为0的行数: {len(bad)} (必须为0)")

with open(SAVE_CSV, 'r') as f:
    line = f.readlines()[1]
    print(f"首行: {line[:100]}...")

print("✅ 下载上传 Nexus")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
模型: /content/drive/MyDrive/Wheat/4_output/A_yolo11x_single/weights/best.pt
测试图: 1025张
0/1025
100/1025
200/1025
300/1025
400/1025
500/1025
600/1025
700/1025
800/1025
900/1025
1000/1025

CSV: /content/drive/MyDrive/Wheat/4_output/submission_A_yolo11x_final.csv
总行数: 346197
宽高为0的行数: 0 (必须为0)
首行: b643c321e44fb9b42aa48efee8f42bee2c6d2f852385d285ce5afa1c608181c1,0,0.145709,0.502998,0.287266,0.1601...
✅ 下载上传 Nexus
